In [1]:
# Task 1
!pip install datasets  # Install the library to load the dataset
from datasets import load_dataset
import pandas as pd

# Load the IMDb dataset
dataset = load_dataset("imdb")

# Convert to DataFrame and sample 5000 rows for efficiency
df = pd.DataFrame(dataset['train'])
df = df.sample(5000, random_state=42)

# Rename columns and map labels (0: negative, 1: positive)
df.rename(columns={'text': 'review', 'label': 'sentiment_label'}, inplace=True)
df['sentiment'] = df['sentiment_label'].map({0: 'negative', 1: 'positive'})

print("Dataset Loaded Successfully!")
print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution:")
print(df['sentiment'].value_counts())
print("\nSample Review:")
print(df['review'].iloc[0][:200] + "...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset Loaded Successfully!
Dataset Shape: (5000, 3)

Class Distribution:
sentiment
negative    2515
positive    2485
Name: count, dtype: int64

Sample Review:
Dumb is as dumb does, in this thoroughly uninteresting, supposed black comedy. Essentially what starts out as Chris Klein trying to maintain a low profile, eventually morphs into an uninspired version...


In [3]:
# Task 2
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download necessary NLTK components (Updated for latest NLTK compatibility)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')  # Required for newer NLTK versions
nltk.download('omw-1.4')    # Required for the lemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Check for empty or non-string inputs
    if not isinstance(text, str):
        return ""

    # 1. Lowercasing
    text = text.lower()
    # 2. Remove URLs and HTML tags
    text = re.sub(r'https?://\S+|www\S+|<.*?>', '', text)
    # 3. Remove Punctuation and Special Characters
    text = re.sub(r'[^a-z\s]', '', text)
    # 4. Tokenization
    tokens = word_tokenize(text)
    # 5. Remove Stopwords and 6. Lemmatization
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(cleaned_tokens)

print("Preprocessing data... this may take a minute.")
# Apply the function
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Preprocessing Complete!")

# Show the first few rows to verify
print(df[['review', 'cleaned_review']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Preprocessing data... this may take a minute.
Preprocessing Complete!
                                                  review  \
6868   Dumb is as dumb does, in this thoroughly unint...   
24016  I dug out from my garage some old musicals and...   
9668   After watching this movie I was honestly disap...   
13640  This movie was nominated for best picture but ...   
14018  Just like Al Gore shook us up with his painful...   

                                          cleaned_review  
6868   dumb dumb thoroughly uninteresting supposed bl...  
24016  dug garage old musical another one favorite wr...  
9668   watching movie honestly disappointed actor sto...  
13640  movie nominated best picture lost casablanca p...  
14018  like al gore shook u painfully honest cleverly...  


In [4]:
# Task 3
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

# Define Features and Target
X = df['cleaned_review']
y = df['sentiment_label'] # 0 for negative, 1 for positive

# Split into Training and Testing sets (80-20)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Bag of Words (BoW)
bow_vec = CountVectorizer(max_features=5000)
X_train_bow = bow_vec.fit_transform(X_train_raw)
X_test_bow = bow_vec.transform(X_test_raw)

# 2. TF-IDF
tfidf_vec = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vec.fit_transform(X_train_raw)
X_test_tfidf = tfidf_vec.transform(X_test_raw)

print("Feature Engineering Complete.")
print(f"BoW Shape: {X_train_bow.shape}")
print(f"TF-IDF Shape: {X_train_tfidf.shape}")

Feature Engineering Complete.
BoW Shape: (4000, 5000)
TF-IDF Shape: (4000, 5000)


In [5]:
# Task 4 & 5
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()
}

performance_metrics = []

# Evaluate each model using TF-IDF features
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    performance_metrics.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

# Display the comparison table
comparison_df = pd.DataFrame(performance_metrics)
print("\n--- Model Performance Comparison (Using TF-IDF) ---")
print(comparison_df)


--- Model Performance Comparison (Using TF-IDF) ---
                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression     0.848   0.825490  0.869835  0.847082
1          Naive Bayes     0.838   0.835417  0.828512  0.831950
2        Decision Tree     0.677   0.668763  0.659091  0.663892
